In [ ]:
import geopandas as gpd
from tqdm import tqdm
import pandas as pd
from pathlib import Path

tqdm.pandas()

#Notes: Code pipeline is this --> LOAD → MERGE → NORMALIZE → OFFSET → EXPORT.


#------Create an output folder if it doesn't exist to save the results.------

#This gets the folder where the script is located, which is assumed to be the "codes" folder in the project structure.
#Made this to work with both Jupyter Notebook and standard Python environments, as __file__ is not defined in Jupyter Notebooks.
try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

#project root directory (one level up from the codes directory)
project_root = codes_dir.parent

#------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

data_folder = project_root / "Data"

print(data_folder)
print(output_folder)

#----------------------------------------------------
# Load all excel files from the Data folder
#----------------------------------------------------
# excel_files = list(data_folder.glob("*.xlsx"))

# dfs = []
# for file in tqdm(excel_files, desc="Loading Excel files"):
#     excel = pd.read_excel(file, header=0)
#     print(f"Loaded {file.name} with {len(excel)} rows.")
#     excel.columns = excel.columns.str.strip().str.lower()

#     excel["source_file"] = file.name  # Add a column to identify the source file
#     dfs.append(excel)

# excel = pd.concat(dfs, ignore_index=True)

excel = pd.read_excel(
    r"C:\School\TieDataProject\Vaylavirasto-data-projekti\Data\tiet-2-3-9.xlsx",
    header=0)
# excel = excel.drop_duplicates()

print("Total rows in excel: ", len(excel))

#----------------------------------------------------
# Load Digiroad geopackage data (KokoSuomi_Digiroad_R_GeoPackage.gpkg)
#----------------------------------------------------
roads = gpd.read_file(
    r"C:\School\TieDataProject\KokoSuomi_Digiroad_R_GeoPackage\KokoSuomi_Digiroad_R_GeoPackage.gpkg",
    layer="DR_LINKKI",
    columns=["TIENUMERO", "TIEOSANRO", "AET", "LET", "geometry"]
)
print("Total rows in roads: ", len(roads))

# print("road ORIGINAL", roads.columns.tolist())
# print("excel ORIGINAL", excel.columns.tolist())

#---------We rename the columns in the both datasets to match for merging.---------
excel.columns = excel.columns.str.strip().str.lower()

excel = excel.rename(columns={
    "tieosanumero": "aosa",
    "aet": "excel_aet",
    "let": "excel_let",
})

# ----------------------------------------------------
# AGGREGATE EXCEL
# ----------------------------------------------------

excel_grouped = excel.groupby(
    ["tie","aosa","ajorata","kaista"],
    as_index=False
).agg({
    "excel_aet": "min",
    "excel_let": "max"
})

#-----From the roads dataset, we select the relevant columns.-------
roads.columns = roads.columns.str.strip().str.lower()
roads = roads[["tienumero","tieosanro","aet","let","geometry"]]

#---------We rename the columns in the roads dataset to match the excel dataset for merging.---------
roads = roads.rename(columns={
    "tienumero": "tie",
    "tieosanro": "aosa",
    "aet": "road_aet",
    "let": "road_let",
})

roads = roads[roads["tie"].isin(excel["tie"])] # Filter roads to only include those that are present in the excel dataset.

print("Unique roads in Excel:", excel["tie"].nunique())
print("Unique roads in roads:", roads["tie"].nunique())

print("road MODDED", roads.columns.tolist())
print("excel MODDED", excel.columns.tolist())

In [ ]:
#Notes: Code pipeline is this --> LOAD → MERGE → NORMALIZE → OFFSET → EXPORT.
#
# ----------------------------------------------------
# DEV MODE
# ----------------------------------------------------

DEV_MODE = True

if DEV_MODE:
    test_roads = [2, 3, 9]  # road number that you wanna test with

    excel = excel[excel["tie"].isin(test_roads)]
    roads = roads[roads["tie"].isin(test_roads)]

LANE_WIDTH = 3.5


# ----------------------------------------------------
# 1. FILTER (HUGE PERFORMANCE BOOST)
# ----------------------------------------------------

# Only keep road segments that exist in Excel
pairs = excel[["tie", "aosa"]].drop_duplicates()

roads = roads.merge(pairs, on=["tie", "aosa"])

# Now merge actual data
merged = roads.merge(excel, on=["tie", "aosa"], how="left")

# FILTER BY SEGMENT OVERLAP
merged = merged[
    (merged["road_aet"] <= merged["excel_aet"]) &
    (merged["road_let"] >= merged["excel_let"])
]

# ----------------------------------------------------
# 2. MINIMAL COLUMNS
# ----------------------------------------------------

merged = merged[[
    "tie", "aosa", "geometry",
    "ajorata", "kaista"
]].copy()

merged["ajorata"] = merged["ajorata"].fillna(0)
merged["kaista"] = merged["kaista"].fillna(11)


# ----------------------------------------------------
# 3. NORMALIZE LANES (VECTOR ONLY)
# ----------------------------------------------------

merged["direction"] = 1

mask0 = merged["ajorata"] == 0
merged.loc[mask0 & (merged["kaista"] >= 20), "direction"] = -1
merged.loc[merged["ajorata"] == 2, "direction"] = -1

merged["lane_index"] = merged["kaista"] % 10
merged.loc[merged["lane_index"] == 0, "lane_index"] = 1


# ----------------------------------------------------
# 4. LANE COUNT
# ----------------------------------------------------

lane_counts = (
    merged.groupby(["tie", "aosa", "direction"])["lane_index"]
    .nunique()
    .rename("lane_count")
)

merged = merged.join(lane_counts, on=["tie", "aosa", "direction"])


# REMOVE DUPLICATES PROPERLY
merged = merged.sort_values(
    ["tie", "aosa", "ajorata", "kaista"]
)

merged = merged.drop_duplicates(
    subset=[
        "tie",
        "aosa",
        "ajorata",
        "kaista",
        "lane_index",
        "direction"
    ]
)
print("After removing duplicates")
print(
    merged.groupby(["tie","aosa","direction"])["lane_index"].nunique()
)

# ----------------------------------------------------
# 5. COMPUTE OFFSET (VECTOR)
# ----------------------------------------------------

center_shift = (merged["lane_count"] - 1) / 2
offset_index = merged["lane_index"] - 1 - center_shift

merged["offset"] = offset_index * LANE_WIDTH * merged["direction"]


# ----------------------------------------------------
# 6. APPLY GEOMETRY OFFSET (CONTROLLED LOOP)
# ----------------------------------------------------

def safe_offset(geom, offset):
    if geom is None or geom.is_empty:
        return geom
    try:
        return geom.parallel_offset(offset, "left", join_style=2)
    except Exception:
        return geom


# Use list comprehension (FASTER than apply)
merged["geometry"] = [
    safe_offset(g, o) for g, o in zip(merged.geometry, merged.offset)
]


# ----------------------------------------------------
# 7. OPTIONAL: explode ONLY if needed
# ----------------------------------------------------

# merged = merged.explode(index_parts=False)
print(
    merged.groupby(["tie", "aosa"]).size().max()
)

print(
    merged.groupby(["tie","aosa","ajorata"])["kaista"].nunique().value_counts()
)


# ----------------------------------------------------
# 8. SAVE
# ----------------------------------------------------

merged.to_file(output_folder / "road_condition.gpkg", driver="GPKG")